# Cross-Attention Two-Stream FUSION (self-contained)

Fuses the features of the two frozen streams and decodes from the joint CTC head:
- appearance: `transformer_feats/{split}.pkl` (Signformer encoder, 256-d)
- skeleton: `skeleton_feats_133/{split}.pkl` (skeleton GCN, 512-d)

**Run All does everything**: extracts the appearance features if missing, then trains
the fusion. Compare the joint WER against: distilled 40.81 test, baseline 41.53,
skeleton 42.61.

In [ ]:
# --- setup path + cuDNN off + estrazione automatica + risoluzione cartelle ---
import os, sys, torch
torch.backends.cudnn.enabled = False   # env cuDNN issue

_here = os.path.abspath('.')
CODE = None
for _k in range(6):
    _b = os.path.abspath(os.path.join(_here, *(['..'] * _k)))
    if os.path.exists(os.path.join(_b, 'csrl_skeleton', 'model.py')):
        CODE = _b; break
assert CODE, 'non trovo code/csrl_skeleton'
DIST = os.path.join(CODE, 'distillation')
FUS = os.path.join(DIST, 'experiments', '02_fusion')
for p in (FUS, os.path.join(CODE, 'csrl_skeleton'), DIST):
    if p not in sys.path:
        sys.path.insert(0, p)
ROOT = os.path.dirname(CODE)

# cartella dei dati 133 (split pickle): prova le due layout note
DATA = None
for _c in (os.environ.get('MSKA_DATA_DIR'),
           os.path.join(ROOT, 'dataset', 'phoenix2014t_133kp'),
           os.path.join(ROOT, 'dataset', 'pose', 'phoenix2014t_133kp')):
    if _c and os.path.exists(os.path.join(_c, 'Phoenix-2014T.train')):
        DATA = _c; break
assert DATA, 'non trovo phoenix2014t_133kp (Phoenix-2014T.train)'
APP = os.path.join(ROOT, 'dataset', 'features', 'transformer_feats')
SKEL = os.path.join(ROOT, 'dataset', 'features', 'skeleton_feats_133')
print('data   :', DATA)
print('skel   :', os.path.exists(os.path.join(SKEL, 'train.pkl')), SKEL)
assert os.path.exists(os.path.join(SKEL, 'train.pkl')), 'manca skeleton_feats_133'

if os.path.exists(os.path.join(APP, 'train.pkl')):
    print('app    : OK', APP)
else:
    print('app    : ASSENTI -> estraggo dal Signformer baseline (~5-10 min)...')
    get_ipython().system(f'python "{os.path.join(DIST, "extract_transformer_feats.py")}"')
    assert os.path.exists(os.path.join(APP, 'train.pkl')), 'estrazione transformer_feats fallita'
    print('app    : estratte ->', APP)

In [ ]:
# --- allena la fusione + valuta (best dev + TEST greedy/beam) ---
# Config DIAGNOSTICA: beta=0 spegne il crossKD (mutual learning) cosi' la testa
# appearance NON viene trascinata dalla skeleton (piu' debole). Le teste ausiliarie
# restano supervisionate (alpha>0). Il log per-epoch ora stampa la WER dev separata:
#   dev joint  |  app  |  skel
# Se la testa APP recupera ~39-42% ma la JOINT resta ~74% -> il problema e' la
# FUSIONE (non le feature). Se anche APP e' ~74% -> bug di allineamento/lunghezze.
from fusion_train import train_fusion

best_dev, ckpt = train_fusion(
    run_dir=os.path.join(ROOT, 'runs', 'fusion'),
    data_dir=DATA, app_feats_dir=APP, skel_feats_dir=SKEL,
    epochs=40, batch_size=16, lr=1e-4, weight_decay=1e-3,
    d_model=256, heads=8, layers=1, dropout=0.2,
    alpha=0.5, beta=0.0, prior_beta=0.0, patience=12,
)
print('best dev WER:', round(best_dev, 2), '->', ckpt)
